In [1]:
# !pip install -U langchain-community
# !pip install -qU langchain-openai
# !pip install --upgrade langchain-community duckduckgo-search
# !pip install langgraph
# !pip install requests pubchempy rdkit-pypi
# !pip install ctx-python
# !pip install biopython
# !pip install langgraph-checkpoint-sqlite
# !pip install faiss-cpu
# !pip install langchain_openai python-dotenv
# !pip install "numpy<2"  # Use NumPy 1.x for compatibility with rdkit-pypi
# !pip install beautifulsoup4
# !pip install elsapy
# !pip install --upgrade numpy
# !pip install docling
# !pip install langchain-core==1.0.0
# !pip install --upgrade langchain
# !pip install sentence-transformers

# !pip install pdfplumber
# !pip install PyMuPDF
# !pip install --upgrade rdkit

In [1]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from pydantic import Field, SecretStr


load_dotenv()

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                 openai_api_key: Optional[str] = None,
                 **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)



/home/chen/GNPS_SIM/agent/db_form_agent_system/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# -------------------- stdlib --------------------
from pathlib import Path
import re, logging, warnings, sqlite3, os, time, random, pickle, requests
import json
import copy
from typing import List, TypedDict

# -------------------- langchain / langgraph --------------------
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage


# -------------------- local tools --------------------
import sys
sys.path.append('../code')
from compound_tool import CompoundDatabaseTool
from searching_tool import FullTextJournalSearchTool
from fulltext_vector_tools import FullTextIndexTool, FullTextQueryTool

# -------------------- logging --------------------
logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(name)s | %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)

# -------------------- constants --------------------
MAX_ITERATIONS = 100

def _vprint(*args, **kwargs):
    # set NODE_LOGS=1 to enable; anything else disables
    if os.getenv("NODE_LOGS", "1") in ("1", "true", "TRUE", "yes", "YES"):
        print(*args, **kwargs)

# -------------------- prompts --------------------
PROMPTS_DIR = Path("/home/chen/GNPS_SIM/agent/db_form_agent_system/prompts")
LITERATURE_PROMPT_PATH = PROMPTS_DIR / "literature_agent.md"
CRITIQUE_PROMPT_PATH   = PROMPTS_DIR / "critique_agent.md"

def _load_prompt(path: Path, fallback: str) -> SystemMessage:
    try:
        return SystemMessage(content=path.read_text(encoding="utf-8"))
    except Exception:
        return SystemMessage(content=fallback.strip())

LITERATURE_AGENT_PROMPT = _load_prompt(
    LITERATURE_PROMPT_PATH,
    fallback="""
You are a meticulous research agent. Use tools when needed. Return JSON with a top-level "synthesis" object
including keys: claim (string), confidence_score (0..1), explanation (string), limitations (string), supporting_items (list).
""",
)

CRITIQUE_AGENT_PROMPT = _load_prompt(
    CRITIQUE_PROMPT_PATH,
    fallback="""
You are a critical evaluator. Read the researcher's output and decide if we should stop. Return JSON with keys:
decision ("approve" | "clarify" | "reject"), stop_reason (APPROVED | NO_NEW_EVIDENCE | STAGNANT_CLAIM | TOOL_UNAVAILABLE | FUTILITY_DETECTED | ""), 
supporting_sources (list), tool_name (string), agreement_score (0..1).
""",
)

# -------------------- model helpers --------------------
def _init_models():
    """
    Initialize research & evaluator chat models from environment variables:
      RESEARCH_MODEL    (default: 'gemini-2.5-flash')
      EVALUATOR_MODEL   (default: 'gpt-4o-mini')
      OPENAI_API_KEY etc. are read by the underlying client(s).
    """
    research_model = ChatOpenRouter(model_name="google/gemini-2.5-flash", temperature=0.0)

    evaluator_model = ChatOpenRouter(model_name="openai/gpt-4o-mini", temperature=0.0)
    return research_model, evaluator_model

# -------------------- tool factory --------------------
def _build_tools(results_base_dir: str) -> List:
    """
    Build your tools with explicit configuration.
    Toggle sources via env:
      INCLUDE_PUBMED=1
      INCLUDE_PREPRINTS=1
      INCLUDE_BIORXIV=1
    """
    ftjs = FullTextJournalSearchTool(
        results_base_dir=results_base_dir,
        include_pubmed=True,
        include_preprints=False,
        include_biorxiv=False,
        include_arxiv=False
    )

    tool_list = [ftjs, FullTextIndexTool(results_base_dir=results_base_dir), FullTextQueryTool(results_base_dir=results_base_dir)]
    if CompoundDatabaseTool is not None:
        tool_list.insert(0, CompoundDatabaseTool())
    return tool_list

# -------------------- build agents --------------------
memory = MemorySaver()
research_model, evaluator_model = _init_models()
tools = _build_tools("/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results")


agent_executor = create_react_agent(
    research_model, tools, prompt=LITERATURE_AGENT_PROMPT, checkpointer=memory
)
critique_agent_executor = create_react_agent(
    evaluator_model, tools, prompt=CRITIQUE_AGENT_PROMPT, checkpointer=memory
)

# -------------------- state definitions --------------------
class StateHistory(TypedDict):
    claim: str
    confidence_score: float
    supporting_sources: List[str]
    tool_name: str | None
    stop_reason: str | None

class AgentState(TypedDict):
    messages: List[HumanMessage | AIMessage]
    original_query: str
    stop_history: List[StateHistory]
    consecutive_no_new_evidence: int
    consecutive_tool_unavailable: int
    last_researcher_json: str | None
    last_critique_json: str | None
    claim_history: List[str]
    confidence_history: List[float]

# -------------------- nodes --------------------
def researcher_node(state: AgentState):
    _vprint("-----Researcher Node Invoked-----")
    last = state["messages"][-1].content if state.get("messages") else ""
    next_query = None

    m = re.search(r"\{[\s\S]*\}", str(last))
    if m:
        try:
            obj = json.loads(m.group(0))
            if (obj.get("decision","") or "").lower() == "clarify":
                next_query = obj.get("next_query")
        except json.JSONDecodeError:
            pass
    if not next_query and "clarify" in str(last).lower():
        try:
            obj = json.loads(m.group(0)) if m else {}
            next_query = obj.get("next_query") or None
        except Exception:
            pass

    if next_query:
        merged = (
            f"Original question:\n{state.get('original_query','')}\n\n"
            f"Follow-up request from reviewer:\n{next_query}\n\n"
            "Answer the follow-up strictly in the context of the original question."
        )
        input_messages = [HumanMessage(content=merged)]
    else:
        input_messages = state["messages"]

    result = agent_executor.invoke({"messages": input_messages})
    return {"messages": state["messages"] + [AIMessage(content=result['messages'][-1].content)]}

def researcher_node(state: AgentState):
    _vprint("-----Researcher Node Invoked-----")

    # If we have a cached critique JSON, build the follow-up message from artifacts.
    crit_json = state.get("last_critique_json")
    prev_research_json = state.get("last_researcher_json") or _last_researcher_json(state.get("messages", []))

    if crit_json and prev_research_json:
        next_query_val = None
        follow_up_str = ""  # Initialize an empty string for the follow-up
        try:
            # Try to parse the critique JSON
            if isinstance(crit_json, str):
                crit_obj = json.loads(crit_json)
            else:
                crit_obj = crit_json  # Already a dict
            # Navigate the structure safely
            critique_section = crit_obj.get("critique", {})
            if not isinstance(critique_section, dict):
                critique_section = {}            
            decision = critique_section.get("decision", "")
            if isinstance(decision, str) and decision.lower() == "clarify":
                next_query_val = critique_section.get("next_query")

            # Check if a next_query was actually found
            if next_query_val:
                query_str = ""
                
                # Handle if LLM returned a list instead of a string
                if isinstance(next_query_val, list):
                    _vprint("[Debug] next_query was a list, converting to string.")
                    # Join list items or convert to readable format
                    query_str = "\n".join(str(item) for item in next_query_val if item)
                # Handle if it's already a string (the intended format)
                elif isinstance(next_query_val, str) and next_query_val.strip():
                    _vprint("[Debug] next_query was already a string.")
                    query_str = next_query_val.strip()
                
                # Only add the follow-up if the query string is not empty
                if query_str:
                    follow_up_str = f"\nReviewer follow-up request:\n{query_str}\n\n"

        except json.JSONDecodeError as e:
            _vprint(f"[WARN] Could not parse critique JSON: {e}")
        except Exception as e:
            _vprint(f"[WARN] Error processing critique for next_query: {e}")

        merged = (
            "Previous researcher output:\n"
            f"{prev_research_json}\n\n"
            "Reviewer critique JSON:\n"
            f"{crit_json}\n\n"
            + follow_up_str  # Safely concatenate the pre-formatted string
        )
        input_messages = [HumanMessage(content=merged)]
    else:
        # First turn or no critique available yet — proceed with the user's original message.
        input_messages = state["messages"]

    # Invoke the agent
    result = agent_executor.invoke({"messages": input_messages})

    # Safely extract the AI message content
    ai_content = ""
    if 'messages' in result and isinstance(result['messages'], list) and result['messages']:
        ai_content = result['messages'][-1].content
    else:
         _vprint("[ERROR] Agent executor did not return expected messages list.")
         ai_content = '{"synthesis": {"claim": "Unclear", "explanation": "Agent execution failed internally. No message was returned.", "confidence_score": 0.0, "limitations": "Agent internal error."}, "traceability": {}, "next_actions": {}}'

    new_ai_message = AIMessage(content=ai_content)
    all_messages = state["messages"] + [new_ai_message]

    return {
        "messages": all_messages,
        # When the researcher speaks, refresh last_researcher_json (for the next loop)
        "last_researcher_json": _last_researcher_json(all_messages),
        # Clear last_critique_json so we only use it once
        "last_critique_json": None,
    }

def critique_node(state: AgentState):
    _vprint("-----Critique Node Invoked-----")

    # Pull the last researcher JSON only (no original query)
    last_researcher_json = _last_researcher_json(state.get("messages", [])) or ""
    if not last_researcher_json:
        # Fallback to the last message content if researcher JSON not found
        last_researcher_json = (state["messages"][-1].content if state.get("messages") else "").strip()

    # Feed ONLY the researcher JSON to the critique agent to avoid echoing original query
    critique_input = f"Researcher output JSON:\n{last_researcher_json}"
    critique_result = critique_agent_executor.invoke({"messages": [HumanMessage(content=critique_input)]})
    critique_content = critique_result['messages'][-1].content
    raw = (critique_content or "").strip()

    parsed: dict | None = None
    try:
        candidate = json.loads(raw)
        if isinstance(candidate, dict):
            parsed = candidate
    except Exception:
        pass
    if parsed is None:
        match = re.search(r"\{[\s\S]*?\}", raw)
        if match:
            try:
                candidate = json.loads(match.group(0))
                if isinstance(candidate, dict):
                    parsed = candidate
            except Exception:
                parsed = None

    stop_reason = None
    supporting_sources: List[str] = []
    tool_name: str | None = None

    if parsed:
        candidate_reason = parsed.get("stop_reason")
        if isinstance(candidate_reason, str) and candidate_reason.strip():
            stop_reason = candidate_reason.strip().upper()
        sources = parsed.get("supporting_sources")
        if isinstance(sources, list):
            supporting_sources = [str(item) for item in sources]
        candidate_tool = parsed.get("tool_name")
        if isinstance(candidate_tool, str) and candidate_tool.strip():
            tool_name = candidate_tool.strip()

    # ---- helpers reused ----
    def _safe_float(value):
        try:
            if isinstance(value, (int, float)):
                return float(value)
            if isinstance(value, str):
                return float(value.replace("–", "-").strip())
        except Exception:
            return None
        return None

    def _extract_claim_and_confidence(messages):
        blob = _last_researcher_json(messages)
        claim = "Unclear"
        confidence = 0.0
        if not blob:
            return claim, confidence
        try:
            obj = json.loads(blob)
            synthesis = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
            claim = (synthesis.get("claim") or "").strip() if isinstance(synthesis, dict) else ""
            candidate_conf = _safe_float(synthesis.get("confidence_score")) if isinstance(synthesis, dict) else None
            if candidate_conf is not None:
                confidence = max(0.0, min(1.0, candidate_conf))
        except Exception:
            pass
        if not claim:
            match_claim = re.search(r'"claim"\s*:\s*"([^"]+)"', blob)
            if match_claim:
                claim = match_claim.group(1).strip()
        if confidence == 0.0:
            match_conf = re.search(r'"confidence_score"\s*:\s*("?)([0-9]*\.?[0-9]+)\1', blob)
            if match_conf:
                cand = _safe_float(match_conf.group(2))
                if cand is not None:
                    confidence = max(0.0, min(1.0, cand))
        return claim or "Unclear", confidence

    claim, confidence = _extract_claim_and_confidence(state.get("messages", []))

    history = copy.deepcopy(state.get("stop_history", []))
    history.append({
        "claim": claim,
        "confidence_score": confidence,
        "supporting_sources": supporting_sources,
        "tool_name": tool_name,
        "stop_reason": stop_reason,
    })

    prev_no_new = state.get("consecutive_no_new_evidence", 0)
    prev_tool_unavailable = state.get("consecutive_tool_unavailable", 0)

    no_new = prev_no_new

    # Track claim and confidence history for consecutive same result detection
    claim_hist = list(state.get("claim_history", []))
    conf_hist = list(state.get("confidence_history", []))
    claim_hist.append(claim if isinstance(claim, str) else "Unclear")
    conf_hist.append(float(confidence) if confidence is not None else 0.0)

    # Check if last 3 iterations have identical claim and confidence
    STABLE_THRESHOLD = 3
    CONFIDENCE_EPSILON = 0.001  # Allow tiny floating point differences
    
    if len(claim_hist) >= STABLE_THRESHOLD and len(conf_hist) >= STABLE_THRESHOLD:
        last_n_claims = claim_hist[-STABLE_THRESHOLD:]
        last_n_confs = conf_hist[-STABLE_THRESHOLD:]
        
        # Normalize claims for comparison (case-insensitive)
        normalized_claims = [c.strip().lower() if isinstance(c, str) else "" for c in last_n_claims]
        
        # Check if all claims are identical
        all_same_claim = len(set(normalized_claims)) == 1
        
        # Check if all confidences are within epsilon
        conf_range = max(last_n_confs) - min(last_n_confs)
        all_same_conf = conf_range <= CONFIDENCE_EPSILON
        
        if all_same_claim and all_same_conf and not stop_reason:
            stop_reason = "CONSECUTIVE_SAME_RESULT"
            _vprint(f"[critique_node] Detected {STABLE_THRESHOLD} consecutive identical results: {last_n_claims[-1]} @ {last_n_confs[-1]:.3f}")
            if history:
                history[-1]["stop_reason"] = stop_reason

    _vprint(f"[critique_node] stop_reason={stop_reason}")

    # Ensure the outgoing critique content is pure JSON (with patched stop_reason if set)
    patched_content = critique_content
    if parsed is not None:
        patched = dict(parsed)
        if stop_reason:
            patched["stop_reason"] = stop_reason
        patched_content = json.dumps(patched, indent=2)

    return {
        "messages": state["messages"] + [AIMessage(content=patched_content)],
        "stop_history": history,
        "consecutive_no_new_evidence": no_new,
        "consecutive_tool_unavailable": prev_tool_unavailable,
        "last_researcher_json": last_researcher_json,
        "last_critique_json": patched_content,
        "claim_history": claim_hist,
        "confidence_history": conf_hist,
    }


def should_continue(state: AgentState):
    """Decide whether to continue the loop using critique stop reasons."""
    last_message_content = state["messages"][-1].content if state.get("messages") else ""
    raw = (last_message_content or "").strip()
    _vprint(f"[should_continue] raw critique output: ...")

    STOP_REASONS = {
        "APPROVED", 
        "NO_NEW_EVIDENCE", 
        "TOOL_UNAVAILABLE",
        "CONSECUTIVE_SAME_RESULT",
    }

    def _normalized_reason(value) -> str | None:
        if isinstance(value, str) and value.strip():
            return value.strip().upper()
        return None

    history = state.get("stop_history") or []
    if history:
        latest_reason = _normalized_reason(history[-1].get("stop_reason"))
        if latest_reason in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (history stop_reason={latest_reason})-----")
            return END

    def _should_stop_from_json(obj: dict) -> bool:
        stop_reason = _normalized_reason(obj.get("stop_reason"))
        if stop_reason in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (stop_reason={stop_reason})-----")
            return True

        decision = obj.get("decision")
        if isinstance(decision, str) and decision.lower() == "approve":
            _vprint("-----Agent Decided to Stop (legacy decision=approve)-----")
            return True

        try:
            score = float(obj.get("agreement_score", 0.0))
            if score > 0.75:
                _vprint(f"-----Agent Decided to Stop (legacy agreement_score={score:.2f} > 0.75)-----")
                return True
        except Exception:
            pass

        return False

    try:
        critique_json = json.loads(raw)
        if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
            return END
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*?\}", raw)
    if match:
        try:
            critique_json = json.loads(match.group(0))
            if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
                _vprint("-----Agent Decided to Stop (extracted JSON)-----")
                return END
        except Exception:
            pass

    low = raw.lower()
    if any(token in low for token in ["approve", "approved"]):
        _vprint("-----Agent Decided to Stop (text match)-----")
        return END

    return "researcher"


# -------------------- workflow --------------------
workflow = StateGraph(AgentState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("critique", critique_node)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "critique")
workflow.add_conditional_edges("critique", should_continue)

app = workflow.compile(checkpointer=memory)

# -------------------- helpers --------------------
def _last_researcher_json(messages):
    """Return the last message content that contains a 'synthesis' JSON, else ''."""
    if not messages:
        return ""
    for msg in reversed(messages):
        content = getattr(msg, "content", "") or ""
        if '"synthesis"' in content:
            return content
    return ""

def determine_final_claim(messages):
    """Read 'synthesis.claim' with minimal normalization."""
    blob = _last_researcher_json(messages)
    if not blob:
        return "Unclear"
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        claim_raw = syn.get("claim")
        if isinstance(claim_raw, str) and claim_raw.strip():
            return claim_raw.strip()
    except Exception:
        pass

    m = re.search(r'"claim"\s*:\s*"([^"]+)"', blob)
    if m:
        candidate = m.group(1).strip()
        if candidate:
            return candidate
    return "Unclear"

def _safe_float(x):
    try:
        if isinstance(x, (int, float)):
            return float(x)
        if isinstance(x, str):
            return float(x.replace("–", "-").strip())
    except Exception:
        return None

def determine_confidence_score(messages):
    """
    Reads numeric 'synthesis.confidence_score' (0..1).
    If missing/unparseable, returns 0.0 (so it won't cross 0.45).
    """
    blob = _last_researcher_json(messages)
    if not blob:
        return 0.0
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        score = syn.get("confidence_score")
        return _safe_float(score)
    except Exception:
        pass
    m = re.search(r'"confidence_score"\s*:\s*([0-9.]+)', blob)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            pass
    return 0.0

def determine_explanation(messages):
    """
    Reads 'synthesis.explanation' from the last researcher JSON.
    Returns empty string if not found.
    """
    blob = _last_researcher_json(messages)
    if not blob:
        return ""
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        explanation = syn.get("explanation")
        if isinstance(explanation, str) and explanation.strip():
            return explanation.strip()
    except Exception:
        pass
    
    # Fallback: try regex extraction
    m = re.search(r'"explanation"\s*:\s*"([^"]+)"', blob)
    if m:
        return m.group(1).strip()
    return ""

/tmp/ipykernel_3168425/896872362.py:104: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(
/tmp/ipykernel_3168425/896872362.py:107: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  critique_agent_executor = create_react_agent(


In [3]:


def main():
    _vprint("\n🧪 Analytical Chemistry Agent Ready!")
    while True:
        try:
            query = input("\n📝 Enter your question (or 'quit' to exit): ")
        except (EOFError, KeyboardInterrupt):
            _vprint("\nExiting.")
            break

        _vprint(f"\n>>> User input query: {query}\n", flush=True)
        if query.lower().strip() == 'quit':
            break

        # per-run session dir for all tools to reuse
        query_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
        query_results_dir = f"/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_{query_timestamp}"
        os.environ["SEARCH_SESSION_DIR"] = query_results_dir

        # optional tracing/project metadata
        os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
        os.environ["LANGCHAIN_PROJECT"] = f"DB Form Agent Run - {query_timestamp}"
        _vprint(f"📁 All results will be saved to: searching_results/run_{query_timestamp}\n")

        init_message = HumanMessage(content=query)
        init_state = {
            'messages': [init_message],
            'original_query': query,
            'stop_history': [],
            'consecutive_no_new_evidence': 0,
            'consecutive_tool_unavailable': 0,
            'claim_history': [],
            'confidence_history': [],
        }

        thread_id = f"qa_session_{int(time.time())}"
        last_event_state = None

        try:
            for event in app.stream(
                init_state,
                config={'configurable': {'thread_id': thread_id}, 'recursion_limit': MAX_ITERATIONS}
            ):
                for v in event.values():
                    last_event_state = v
                    try:
                        content = v['messages'][-1].content
                        try:
                            obj = json.loads(content)
                            _vprint(json.dumps(obj, indent=2))
                        except json.JSONDecodeError:
                            _vprint(content)
                    except Exception:
                        _vprint(str(v))
        except GraphRecursionError as ge:
            _vprint(f"[GraphRecursionError] {ge}")
        except KeyboardInterrupt:
            _vprint("\nInterrupted by user.")
            break
        except Exception as e:
            _vprint(f"[ERROR] Agent execution failed: {e}")

        claim = 'Unclear'
        score = 0.0
        explanation = ""
        critique_decision = None
        critique_json = {}

        if last_event_state and last_event_state.get('messages'):
            claim = determine_final_claim(last_event_state['messages'])
            score = determine_confidence_score(last_event_state['messages']) or 0.0
            explanation = determine_explanation(last_event_state['messages'])

        display_label = claim.strip() if isinstance(claim, str) and claim.strip() else "Unclear"

        final_stop_reason = None
        if last_event_state and last_event_state.get('stop_history'):
            final_stop_reason = (last_event_state['stop_history'][-1] or {}).get('stop_reason')

        _vprint(f"\n=== FINAL DECISION: {display_label} (confidence {score:.2f}) ===")
        if final_stop_reason:
            _vprint(f"    stop reason: {final_stop_reason}")
        if explanation:
            _vprint(f"    explanation: {explanation}")

        if last_event_state and last_event_state.get('messages'):
            critique_msg = last_event_state['messages'][-1]
            critique_raw = getattr(critique_msg, 'content', '') or ''
            try:
                critique_json = json.loads(critique_raw)
            except Exception:
                match_json = re.search(r"\{[\s\S]*?\}", critique_raw)
                if match_json:
                    try:
                        critique_json = json.loads(match_json.group(0))
                    except Exception:
                        critique_json = {}
            if isinstance(critique_json, dict):
                decision_val = critique_json.get('decision')
                if isinstance(decision_val, str):
                    critique_decision = decision_val.strip().lower()
        if critique_decision:
            _vprint(f"    critique decision: {critique_decision}")
        _vprint(f"    researcher claim: {claim}")


if __name__ == '__main__':
    main()


🧪 Analytical Chemistry Agent Ready!

>>> User input query: is the compound DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"

📁 All results will be saved to: searching_results/run_20251028_174445

-----Researcher Node Invoked-----


2025-10-28 10:44:48,420 | INFO | Reusing existing session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:44:48,421 | INFO | Starting search #1/3 for query: 'DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) AND human AND (endogenous OR metabolite)'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:44:48,422 | INFO | [1/3] Searching PubMed/PMC…
2025-10-28 10:44:48,755 | INFO | Starting search #2/3 for query: 'di-Meadoyl glycerol AND human AND (endogenous OR metabolite)'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:44:48,756 | INFO | [2/3] Searching PubMed/PMC…
2025-10-28 10:45:39,255 | INFO | Starting search #3/3 for query: 'diacylglycerol AND (20:3 n-9 OR Mead OR eicosatrienoyl) AND human AND (endogenous OR metabolite)'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174

{
  "synthesis": {
    "claim": "Unclear",
    "explanation": "While diacylglycerols (DAGs) are recognized as endogenous human metabolites and lipidomics studies frequently identify various DAG species in human samples, direct evidence specifically identifying DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) or di-Meadoyl glycerol as an endogenous human metabolite is not explicitly present in the retrieved snippets. The searches yielded general information about diacylglycerols and their presence in human metabolomic profiles, but not the specific compound in question.",
    "confidence_score": 0.4,
    "limitations": "The retrieved literature discusses diacylglycerols as a class and mentions various fatty acid chain lengths and double bond counts, but does not specifically confirm the endogenous nature of DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) in humans. The term 'Mead acid' (20:3 n-9) is associated with this compound, and while Mead acid itself is an endogenous fatty acid, its specific incor

2025-10-28 10:46:05,412 | INFO | Reusing existing session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:46:05,412 | INFO | Starting search #1/3 for query: 'DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) human endogenous metabolite'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:46:05,413 | INFO | [1/3] Searching PubMed/PMC…
2025-10-28 10:46:05,772 | INFO | Starting search #2/3 for query: 'di-Meadoyl glycerol human endogenous metabolite'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:46:05,773 | INFO | [2/3] Searching PubMed/PMC…
2025-10-28 10:46:54,998 | INFO | Starting search #3/3 for query: 'diacylglycerol AND (20:3 n-9 OR Mead OR eicosatrienoyl) AND human AND (endogenous OR metabolite)'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:46:54,99

{
  "synthesis": {
    "claim": "Unclear",
    "explanation": "Despite further searching, direct evidence specifically identifying DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) or di-Meadoyl glycerol as an endogenous human metabolite remains elusive. The retrieved literature consistently confirms that diacylglycerols (DAGs) as a class are endogenous human metabolites and are routinely profiled in human studies. However, none of the snippets provide explicit confirmation of the specific compound in question being identified in human samples.",
    "confidence_score": 0.4,
    "limitations": "The current literature search confirms the endogenous nature of diacylglycerols as a class in humans and their profiling in various studies. However, there is no direct evidence found that specifically identifies DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) or di-Meadoyl glycerol as an endogenous human metabolite. The searches for the specific compound or its common aliases did not yield any direct hits, sugge

2025-10-28 10:47:20,495 | INFO | Reusing existing session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:47:20,496 | INFO | Starting search #1/3 for query: 'DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) human biofluids'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:47:20,497 | INFO | [1/3] Searching PubMed/PMC…
2025-10-28 10:47:20,835 | INFO | Starting search #2/3 for query: 'di-Meadoyl glycerol human tissues'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:47:20,836 | INFO | [2/3] Searching PubMed/PMC…
2025-10-28 10:48:10,473 | INFO | Starting search #3/3 for query: 'diacylglycerol 20:3 n-9 human endogenous'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251028_174445
2025-10-28 10:48:10,474 | INFO | [3/3] Searching PubMed/PMC…
2025-10-28 10:48:10,851 | INFO | Found 25 r

{
  "synthesis": {
    "claim": "Unclear",
    "explanation": "Despite further targeted searching, direct evidence specifically identifying DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) or di-Meadoyl glycerol as an endogenous human metabolite in biofluids or tissues remains elusive. The retrieved literature confirms that diacylglycerols (DAGs) as a class are endogenous human metabolites and are routinely profiled in human studies. However, none of the snippets provide explicit confirmation of the specific compound in question being identified in human samples. The searches for the specific compound or its common aliases did not yield any direct hits.",
    "confidence_score": 0.4,
    "limitations": "The current literature search confirms the endogenous nature of diacylglycerols as a class in humans and their profiling in various studies. However, there is no direct evidence found that specifically identifies DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) or di-Meadoyl glycerol as an endogenous hum

In [ ]:
import pandas as pd

df = pd.read_csv("/home/chen/GNPS_SIM/agent/db_form_agent_system/notebooks/quantified_metabolites.csv")

# get 100 rows with 80% endogenous
df_endogenous = df[df['Endogenous']=='Yes'].sample(n=27, random_state=42)
df_non_endogenous = df[df['Endogenous']=='No'].sample(n=3, random_state=42)
df_sampled = pd.concat([df_endogenous, df_non_endogenous]).reset_index(drop=True)

input_command = df_sampled["Name"].to_list()

input_command[:2]


['DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0)',
 'TG(16:0/16:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))']

In [ ]:
is the compound DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"

In [ ]:
# def batch_sequential(query_ls):
#     for query in query_ls:
#         query = f'''is compound {query} endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"'''
#         _vprint(f"\n📝 Processing question: {query}\n", flush=True)

#         # per-run session dir for all tools to reuse
#         query_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
#         query_results_dir = f"/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_{query_timestamp}"
#         os.environ["SEARCH_SESSION_DIR"] = query_results_dir

#         # optional tracing/project metadata
#         os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
#         os.environ["LANGCHAIN_PROJECT"] = f"DB Form Agent Run - {query_timestamp}"
#         _vprint(f"📁 All results will be saved to: searching_results/run_{query_timestamp}\n")

#         # allow users to force label set with:  answer: "Endogenous" "Exogenous"
#         requested_labels: list[str] = []
#         directive = query
#         match = re.search(r"(?i)answer:\s*(.*)", query)
#         if match:
#             raw_segment = match.group(1)
#             candidates = [label.strip() for label in re.findall(r'"([^"]+)"', raw_segment)]
#             for label in candidates:
#                 if label and label not in requested_labels:
#                     requested_labels.append(label)
#             if requested_labels:
#                 label_text = ", ".join(f'"{label}"' for label in requested_labels)
#                 directive = (
#                     f"{query}\n\nInstruction: ensure synthesis.claim exactly matches one of the user-specified answer options: {label_text}. "
#                     "Select the option best supported by the evidence."
#                 )

#         init_message = HumanMessage(content=directive)
#         init_state = {
#             'messages': [init_message],
#             'original_query': query,
#             'stop_history': [],
#             'consecutive_no_new_evidence': 0,
#             'consecutive_stagnant_claim': 0,
#             'consecutive_tool_unavailable': 0,
#             'last_critique_signature': None,
#             'consecutive_clarify_without_progress': 0,
#         }

#         thread_id = f"qa_session_{int(time.time())}"
#         last_event_state = None

#         try:
#             for event in app.stream(
#                 init_state,
#                 config={'configurable': {'thread_id': thread_id}, 'recursion_limit': MAX_ITERATIONS}
#             ):
#                 for v in event.values():
#                     last_event_state = v
#                     try:
#                         content = v['messages'][-1].content
#                         try:
#                             obj = json.loads(content)
#                             _vprint(json.dumps(obj, indent=2))
#                         except json.JSONDecodeError:
#                             _vprint(content)
#                     except Exception:
#                         _vprint(str(v))
#         except GraphRecursionError as ge:
#             _vprint(f"[GraphRecursionError] {ge}")
#         except KeyboardInterrupt:
#             _vprint("\nInterrupted by user.")
#             break
#         except Exception as e:
#             _vprint(f"[ERROR] Agent execution failed: {e}")

#         claim = 'Unclear'
#         score = 0.0
#         critique_decision = None

#         if last_event_state and last_event_state.get('messages'):
#             claim = determine_final_claim(last_event_state['messages'])
#             score = determine_confidence_score(last_event_state['messages']) or 0.0

#         display_label = claim.strip() if isinstance(claim, str) and claim.strip() else "Unclear"

#         _vprint(f"\n=== FINAL DECISION: {display_label} (confidence {score:.2f}) ===")
#         if critique_decision:
#             _vprint(f"    critique decision: {critique_decision}")
#         _vprint(f"    researcher claim: {claim}")

#         return display_label, score


In [ ]:
# batch_sequential(input_command[:2])

In [ ]:
# import pandas as pd
# from tqdm import tqdm


# def is_endogenous_compound(compound_name: str) -> str:
#     prompt = f"Is the compound '{compound_name}' an endogenous metabolite from human? Answer 'Endogenous', 'Exogenous'."

#     query_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
#     query_results_dir = f"/home/chen/proj/db_form_agent_system/searching_results/run_{query_timestamp}"
#     os.environ["SEARCH_SESSION_DIR"] = query_results_dir

#     requested_labels: list[str] = []
#     match = re.search(r"(?i)answer:\s*(.*)", prompt)
#     if match:
#         raw_segment = match.group(1)
#         candidates = [label.strip() for label in re.findall(r'"([^\"]+)"', raw_segment)]
#         for label in candidates:
#             if label and label not in requested_labels:
#                 requested_labels.append(label)
#         if requested_labels:
#             label_text = ", ".join(f'"{label}"' for label in requested_labels)
#             prompt = (
#                 f"{prompt}\n\nInstruction: ensure synthesis.claim exactly matches one of the user-specified answer options: {label_text}. "
#                 "Select the option best supported by the evidence."
#             )

#     init_message = HumanMessage(content=prompt)
#     init_state = {
#         'messages': [init_message],
#         'original_query': prompt,
#         'stop_history': [],
#         'consecutive_no_new_evidence': 0,
#         'consecutive_stagnant_claim': 0,
#         'consecutive_tool_unavailable': 0,
#         'last_critique_signature': None,
#         'consecutive_clarify_without_progress': 0,
#     }

#     thread_id = f"qa_session_{int(time.time())}"
#     last_event_state = None
#     try:
#         for event in app.stream(init_state, config={'configurable': {'thread_id': thread_id}, 'recursion_limit': 20}):
#             for v in event.values():
#                 last_event_state = v
#                 try:
#                     _vprint(v['messages'][-1].content)
#                 except Exception:
#                     _vprint(str(v))
#     except GraphRecursionError:
#         _vprint("-----Reached max iteration limit; stopping further processing-----")
#         return "Reached max iteration"

#     claim = 'Unclear'

#     if last_event_state and last_event_state.get('messages'):
#         claim = determine_final_claim(last_event_state['messages'])

#     return claim


# df = pd.read_csv("/data/chen/data/hmdb/metabolites.csv")

# # get 100 rows with 80% endogenous
# df_endogenous = df[df['Endogenous']=='Yes'].sample(n=45, random_state=42)
# df_non_endogenous = df[df['Endogenous']=='No'].sample(n=5, random_state=42)
# df_sampled = pd.concat([df_endogenous, df_non_endogenous]).reset_index(drop=True)

# tqdm.pandas(desc="Checking compounds") 
# df_sampled['Endogenous_LLM'] = df_sampled['Name'].progress_apply(is_endogenous_compound)
# df_sampled[['Name', 'Endogenous', 'Endogenous_LLM']]


In [ ]:
df_sampled[['Name', 'Endogenous', 'Endogenous_LLM']]

In [ ]:
df_sampled

In [ ]:
import pandas as pd
df = pd.read_csv('/data/chen/data/hmdb/metabolites_endo_exo.csv')

df['Endogenous_LLM'] = df['Name'].apply(is_endogenous_compound)
df[['Name', 'Origin', 'Endogenous_LLM']]